# Практика 3 — Gubeev

Самостоятельная работа: Iris и Wine, две ветки DVC, слияние, чистое восстановление и три версии ClearML.

DVC и ClearML выполнены. Реальные коммиты, Dataset IDs, граф родителей и результаты проверок сохранены в REPORT.md и evidence/. Этот notebook читает сохранённые результаты и проверяет данные без повторного создания серверных объектов.

## Окружение

Выберите Python 3.12 из ../.venv. Все рабочие пути задаются относительно данного notebook; ячейки не меняют историю Git.

In [1]:
from pathlib import Path
import sys, json, subprocess, importlib.metadata as metadata
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / "prepare_data.py").exists():
    ROOT = ROOT / "mlops-data-Gubeev"
sys.path.insert(0, str(ROOT.resolve()))
from prepare_data import prepare
from check_data import check_data
print("Python:", sys.version.split()[0])
for package in ["dvc", "clearml", "scikit-learn", "pandas"]:
    print(package, metadata.version(package))

Python: 3.12.10
dvc 3.59.2
clearml 1.18.0
scikit-learn 1.6.1
pandas 2.2.3


## Фактическое итоговое состояние

Iris: 150 строк, исходные признаки и target плюс sepal_area/petal_area. Wine: 178 строк, 13 исходных признаков и target. Валидация проверяет также значения и порядок строк.

In [2]:
summary = check_data(ROOT / "data", "merged")
display(pd.DataFrame(summary))

,path,sha256,rows,column_count,columns,missing_values
0,data/iris.csv,8228a6345ef84c613f21e4dca934ad87fc9d8629db248f...,150,7,"[sepal length (cm), sepal width (cm), petal le...",0
1,data/wine.csv,b72adfd45219c12d4af882f76b4c67df0a44922d0bac0f...,178,14,"[alcohol, malic_acid, ash, alcalinity_of_ash, ...",0


## Детерминированная генерация

Генерируем отдельные временные данные, сохраняя итоговый DVC workspace.

In [3]:
import tempfile
with tempfile.TemporaryDirectory() as temporary:
    directory = Path(temporary)
    prepare("base", directory)
    iris_bytes = (directory / "iris.csv").read_bytes()
    prepare("base", directory)
    assert iris_bytes == (directory / "iris.csv").read_bytes()
    check_data(directory, "base")
    prepare("wine", directory)
    assert iris_bytes == (directory / "iris.csv").read_bytes()
    check_data(directory, "wine")
    prepare("iris_features", directory)
    display(pd.DataFrame(check_data(directory, "merged")))
print("Повторная генерация базы и сохранение соседних CSV проверены")

,path,sha256,rows,column_count,columns,missing_values
0,tmpgtp382vw/iris.csv,8228a6345ef84c613f21e4dca934ad87fc9d8629db248f...,150,7,"[sepal length (cm), sepal width (cm), petal le...",0
1,tmpgtp382vw/wine.csv,b72adfd45219c12d4af882f76b4c67df0a44922d0bac0f...,178,14,"[alcohol, malic_acid, ash, alcalinity_of_ash, ...",0


Повторная генерация базы и сохранение соседних CSV проверены


## История двух веток и merge

Git хранит data.dvc и код. Объекты CSV получаются из DVC remote. Между ветками в выполненном сценарии использовался dvc checkout.

In [4]:
dvc_evidence = json.loads((ROOT / "evidence/dvc.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{ "state": state, **{key: value for key, value in record.items() if key != "tables"}} for state, record in dvc_evidence["versions"].items()]))
print(dvc_evidence["merge_observation"])
print(subprocess.run(["git", "log", "--graph", "--all", "--oneline", "-12"], cwd=ROOT, text=True, encoding="utf-8", check=True, capture_output=True).stdout)

,state,commit,ref,directory_md5,metadata_sha256
0,base,1e91d10efc3bb56147037f5cefd37575916902c3,data-v1,68694d6e2fc4bf7a7da11d2b0d489e1e.dir,3aa2128aa15e4da33d4f63faa9c38edb24d1b02d1cab7c...
1,wine,7e806b22116f06bbdaf7e14545780829571cacec,feature/wine,cd692bf1d9bad578ff6e9bdaf5fff033.dir,1363389a0a3865e646f3d2b8cb1e71bd518af1f5fa2197...
2,iris_features,cbe9f4edbfd0061dacee5b3b8bbfe2f4cf8df091,feature/iris-features before merge (data-merge...,84fad98472f78577bceb6d442c1f794d.dir,c2574a6022983aaaeedd6a3722cfd498fb901e2434a1af...
3,merged,c6a18cf11c2dc94f3e5939b23d6d5381873ce5e1,main; data-merged,d23e2a33f1aa86af524072021da153ed.dir,53ae84294d6d0310b6dedf66e175c94afae61db8a05f6e...


DVC merge driver автоматически объединил оба состояния при git merge. Конфликта не возникло; после checkout итоговые SHA-256 Iris и Wine совпали с соответствующими ветками.


* 83697ed Update documentation after removing temporary practice artifacts
* af178b1 Record verified DVC recovery and executed practice notebook
*   c6a18cf Merge Wine and Iris features using DVC merge driver
|\  
| * 7e806b2 Add Wine alongside unchanged base Iris
* | cbe9f4e Add sepal and petal area features to Iris
|/  
* 1e91d10 Track deterministic base Iris dataset with DVC



## Проверка чистого восстановления

Клон создавался без data и cache. После dvc pull сравнивались байтовые хэши с итоговой версией исходного репозитория.

In [5]:
restored = dvc_evidence["clone_tables"]
assert [row["sha256"] for row in restored] == [row["sha256"] for row in summary]
display(pd.DataFrame(restored))
print("DVC status при проверке:", dvc_evidence["status"])

,path,sha256,rows,column_count,columns,missing_values
0,data/iris.csv,8228a6345ef84c613f21e4dca934ad87fc9d8629db248f...,150,7,"[sepal length (cm), sepal width (cm), petal le...",0
1,data/wine.csv,b72adfd45219c12d4af882f76b4c67df0a44922d0bac0f...,178,14,"[alcohol, malic_acid, ash, alcalinity_of_ash, ...",0


DVC status при проверке: {}


## ClearML: две дочерние ветви от базы

Проект MIIT/MLOps/practice_3/Gubeev, имя sklearn-tabular-Gubeev.

```text
BASE_ID 1.0.0: iris.csv
├── WINE_ID 1.1.0: iris.csv + wine.csv
└── IRIS_FEATURES_ID 2.0.0: изменённый iris.csv, без Wine
```

Схема подтверждена get_dependency_graph. Все три версии имеют статус completed. После создания дочерних версий все данные повторно скачаны по ID с новым пустым SDK cache; SHA-256 совпали с независимой генерацией. Credentials в notebook не используются.

In [6]:
observed = json.loads((ROOT / "evidence/clearml.json").read_text(encoding="utf-8"))
assert len(observed["versions"]) == 3
assert observed["fresh_cache"] is True
base_id = observed["ids"]["BASE_ID"]
wine_id = observed["ids"]["WINE_ID"]
display(pd.DataFrame([{
    "version": version["version"], "id": version["id"],
    "status": version["status"], "parents": version["parents"],
    "files": version["files"],
} for version in observed["versions"]]))
for version in observed["versions"]:
    assert version["status"] == "completed"
    assert version["finalized"] is True
    expected_parents = [] if version["key"] == "BASE_ID" else [base_id]
    assert version["graph"][version["id"]] == expected_parents
    if version["key"] == "IRIS_FEATURES_ID":
        assert wine_id not in version["graph"]
    print(version["version"], version["graph"], version["changes"])
    display(pd.DataFrame(version["tables"]))
print("Три версии completed, две ветви от базы и проверка с пустым cache подтверждены")

,version,id,status,parents,files
0,1.0.0,77ab3b41f30a48a7b4da01a60e705aba,completed,[],[iris.csv]
1,1.1.0,fd4384d1b03242269ec15a584a543499,completed,[77ab3b41f30a48a7b4da01a60e705aba],"[iris.csv, wine.csv]"
2,2.0.0,23306db43cc441fd8eb27f66502bf1a1,completed,[77ab3b41f30a48a7b4da01a60e705aba],[iris.csv]


1.0.0 {'77ab3b41f30a48a7b4da01a60e705aba': []} {'added': ['iris.csv'], 'modified': [], 'removed': []}


,path,sha256,rows,column_count,columns,missing_values
0,1.0.0/iris.csv,acb1d0e09be856a27d25798722572591b3ec55df238235...,150,5,"[sepal length (cm), sepal width (cm), petal le...",0


1.1.0 {'77ab3b41f30a48a7b4da01a60e705aba': [], 'fd4384d1b03242269ec15a584a543499': ['77ab3b41f30a48a7b4da01a60e705aba']} {'added': ['wine.csv'], 'modified': [], 'removed': []}


,path,sha256,rows,column_count,columns,missing_values
0,1.1.0/iris.csv,acb1d0e09be856a27d25798722572591b3ec55df238235...,150,5,"[sepal length (cm), sepal width (cm), petal le...",0
1,1.1.0/wine.csv,b72adfd45219c12d4af882f76b4c67df0a44922d0bac0f...,178,14,"[alcohol, malic_acid, ash, alcalinity_of_ash, ...",0


2.0.0 {'23306db43cc441fd8eb27f66502bf1a1': ['77ab3b41f30a48a7b4da01a60e705aba']} {'added': [], 'modified': ['iris.csv'], 'removed': []}


,path,sha256,rows,column_count,columns,missing_values
0,2.0.0/iris.csv,8228a6345ef84c613f21e4dca934ad87fc9d8629db248f...,150,7,"[sepal length (cm), sepal width (cm), petal le...",0


Три версии completed, две ветви от базы и проверка с пустым cache подтверждены


## Отчёт

Сравнительная таблица, реальные коммиты, Dataset IDs, ссылки на страницы ClearML, хэши всех состояний и ответы на семь вопросов — в [REPORT.md](REPORT.md). Для повторного скачивания существующих версий используйте clearml_versions.py --execute --inspect-only --fresh-cache согласно README.md.